In [1]:
using DynamicPolynomials
@polyvar x[1:3]

(Variable{DynamicPolynomials.Commutative{DynamicPolynomials.CreationOrder}, Graded{LexOrder}}[x₁, x₂, x₃],)

In [2]:
f = [-x[1]^3 - x[1] * x[3]^2,
     -x[2] - x[1]^2 * x[2],
     -x[3] - 3x[3] / (x[3]^2 + 1) + 3x[1]^2 * x[3]]

3-element Vector{RationalPoly{Polynomial{DynamicPolynomials.Commutative{DynamicPolynomials.CreationOrder}, Graded{LexOrder}, Int64}, Polynomial{DynamicPolynomials.Commutative{DynamicPolynomials.CreationOrder}, Graded{LexOrder}, Int64}}}:
 (-x₁x₃² - x₁³) / (1)
 (-x₂ - x₁²x₂) / (1)
 (-4x₃ - x₃³ + 3x₁²x₃ + 3x₁²x₃³) / (1 + x₃²)

In [4]:
using SumOfSquares
using MosekTools
solver = optimizer_with_attributes(Mosek.Optimizer, MOI.Silent() => true)
model = SOSModel(solver);

In [5]:
monos = x.^2

3-element Vector{Monomial{DynamicPolynomials.Commutative{DynamicPolynomials.CreationOrder}, Graded{LexOrder}}}:
 x₁²
 x₂²
 x₃²

In [6]:
@variable(model, V, Poly(monos))

(_[1])x₃² + (_[2])x₂² + (_[3])x₁²

In [7]:
# @constraint(model, V >= sum(x.^2))
@constraint(model, V >= 0)

(_[1])x₃² + (_[2])x₂² + (_[3])x₁² is SOS

In [8]:
using LinearAlgebra # Needed for `dot`
dVdt = dot(differentiate(V, x), f)

((-8 _[1])x₃² + (-2 _[2])x₂² + (-2 _[1])x₃⁴ + (-2 _[2])x₂²x₃² + (-2 _[3] + 6 _[1])x₁²x₃² + (-2 _[2])x₁²x₂² + (-2 _[3])x₁⁴ + (-2 _[3] + 6 _[1])x₁²x₃⁴ + (-2 _[2])x₁²x₂²x₃² + (-2 _[3])x₁⁴x₃²) / (1 + x₃²)

In [9]:
P = dVdt.num

(-8 _[1])x₃² + (-2 _[2])x₂² + (-2 _[1])x₃⁴ + (-2 _[2])x₂²x₃² + (-2 _[3] + 6 _[1])x₁²x₃² + (-2 _[2])x₁²x₂² + (-2 _[3])x₁⁴ + (-2 _[3] + 6 _[1])x₁²x₃⁴ + (-2 _[2])x₁²x₂²x₃² + (-2 _[3])x₁⁴x₃²

In [10]:
@constraint(model, P <= 0)

(8 _[1])x₃² + (2 _[2])x₂² + (2 _[1])x₃⁴ + (2 _[2])x₂²x₃² + (-6 _[1] + 2 _[3])x₁²x₃² + (2 _[2])x₁²x₂² + (2 _[3])x₁⁴ + (-6 _[1] + 2 _[3])x₁²x₃⁴ + (2 _[2])x₁²x₂²x₃² + (2 _[3])x₁⁴x₃² is SOS

In [11]:
JuMP.optimize!(model)

In [12]:
JuMP.primal_status(model)

FEASIBLE_POINT::ResultStatusCode = 1

In [13]:
value(V)

0.1796505015490793x₃² + 0.5283482357719033x₂² + 0.8129102650095659x₁²